<a href="https://colab.research.google.com/github/gspark0714-cell/python_basic/blob/main/day4_%EC%97%B0%EA%B4%80%EB%B6%84%EC%84%9D.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import warnings
import logging
warnings.filterwarnings('ignore')
logging.disable(logging.WARNING)

In [4]:
!pip install mlxtend -q
!apt-get install -y fonts-nanum -q
import matplotlib
matplotlib.font_manager._load_fontmanager(try_read_cache=False)

Reading package lists...
Building dependency tree...
Reading state information...
The following NEW packages will be installed:
  fonts-nanum
0 upgraded, 1 newly installed, 0 to remove and 7 not upgraded.
Need to get 10.3 MB of archives.
After this operation, 34.1 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 fonts-nanum all 20200506-1 [10.3 MB]
Fetched 10.3 MB in 1s (7,334 kB/s)
Selecting previously unselected package fonts-nanum.
(Reading database ... 118194 files and directories currently installed.)
Preparing to unpack .../fonts-nanum_20200506-1_all.deb ...
Unpacking fonts-nanum (20200506-1) ...
Setting up fonts-nanum (20200506-1) ...
Processing triggers for fontconfig (2.13.1-4.2ubuntu5) ...


In [5]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('군부대_탈영예측_연관분석.csv', encoding='utf-8-sig')
df = df.drop(columns=['병사ID'])

print("=== 데이터 크기 ===")
print(f"병사 수: {df.shape[0]}명, 변수 수: {df.shape[1]}개")
print(f"\n=== 탈영 발생률 ===")
print(f"탈영 발생: {df['탈영발생'].sum()}명 ({df['탈영발생'].mean()*100:.1f}%)")
print(f"탈영 미발생: {(df['탈영발생']==0).sum()}명 ({(1-df['탈영발생'].mean())*100:.1f}%)")

=== 데이터 크기 ===
병사 수: 500명, 변수 수: 9개

=== 탈영 발생률 ===
탈영 발생: 185명 (37.0%)
탈영 미발생: 315명 (63.0%)


In [6]:
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

# 값이 1인 항목만 트랜잭션에 포함
transactions = []
for _, row in df.iterrows():
    items = [col for col in df.columns if row[col] == 1]
    transactions.append(items)

# 원-핫 인코딩
te = TransactionEncoder()
te_array = te.fit_transform(transactions)
df_encoded = pd.DataFrame(te_array, columns=te.columns_)

# 빈발 아이템셋 추출
frequent_itemsets = apriori(df_encoded, min_support=0.05, use_colnames=True)

# 연관규칙 생성
rules = association_rules(frequent_itemsets, metric='confidence', min_threshold=0.3)

print(f"전체 연관규칙 수: {len(rules)}")

전체 연관규칙 수: 3179


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [7]:
rules_target = rules[rules['consequents'].apply(lambda x: '탈영발생' in x)].copy()
rules_target['원인조합'] = rules_target['antecedents'].apply(lambda x: ' + '.join(list(x)))
rules_target = rules_target.sort_values('lift', ascending=False).reset_index(drop=True)

print(f"탈영 촉진 조합 수: {len(rules_target)}개")
rules_target[['원인조합', 'support', 'confidence', 'lift']].head(20)

탈영 촉진 조합 수: 1100개


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

,원인조합,support,confidence,lift
0,상담거부 + 외출제한 + 가족연락두절,0.068,0.507463,3.475772
1,상담거부 + 무단이탈경고 + 가족연락두절,0.068,0.548387,3.470804
2,훈련성적저하 + 무단이탈경고,0.068,0.459459,3.428802
3,상담거부 + 가족연락두절,0.068,0.400000,3.333333
4,외출제한 + 훈련성적저하 + 무단이탈경고,0.068,0.566667,3.333333
5,외출제한 + 훈련성적저하,0.068,0.409639,3.303537
6,상담거부 + 외출제한 + 무단이탈경고 + 가족연락두절,0.068,0.666667,3.300330
7,상담거부 + 외출제한 + 가족연락두절,0.068,0.507463,3.252966
8,동료갈등 + 훈련성적저하 + 가족연락두절,0.060,0.697674,3.229974
9,상담거부 + 훈련성적저하 + 무단이탈경고,0.068,0.755556,3.228870


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [8]:
rules_low = rules_target.sort_values('lift', ascending=True).copy()

# 원인조합 기준으로 중복 제거 (lift 가장 낮은 것만 유지)
rules_low = rules_low.drop_duplicates(subset=['원인조합']).reset_index(drop=True)

# lift < 1.0 이 없으면 하위 10개 출력
if len(rules_low[rules_low['lift'] < 1.0]) == 0:
    print("※ lift < 1.0 조합 없음 → 상대적으로 탈영 영향이 적은 조합 Top 10")
    rules_low = rules_low.head(10)

rules_low[['원인조합', 'support', 'confidence', 'lift']]

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

※ lift < 1.0 조합 없음 → 상대적으로 탈영 영향이 적은 조합 Top 10


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,원인조합,support,confidence,lift
0,수면장애,0.102,0.318750,1.410398
1,동료갈등,0.108,0.329268,1.630041
2,훈련성적저하,0.108,0.362416,1.647346
3,상담거부,0.102,0.356643,1.938279
4,동료갈등 + 수면장애,0.052,0.406250,2.011139
5,훈련성적저하 + 식사거부,0.056,0.451613,2.052786
6,훈련성적저하 + 수면장애,0.052,0.456140,2.073365
7,식사거부,0.084,0.336000,2.100000
8,외출제한,0.160,0.441989,2.104709
9,상담거부 + 훈련성적저하,0.066,0.471429,2.142857


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

In [ ]:
bb